# 00 — Setup and sample data

**LREC-COLING 2026 tutorial — LLM-as-annotator pipelines**

This notebook prepares a Colab-friendly workspace and creates a small multilingual toy dataset used by the following notebooks.

The examples are **pedagogical placeholders**, not an authoritative linguistic resource. Replace them with your project data before any real experiment.

In [ ]:
# In Colab, most packages below are already available. This keeps the notebooks portable.
!pip -q install jsonschema scikit-learn

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_DIR = Path('/content/lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
PROMPT_DIR = PROJECT_DIR / 'prompts'

for d in [DATA_DIR, SCHEMA_DIR, OUTPUT_DIR, PROMPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project directory:', PROJECT_DIR)

## Data model

Each row corresponds to one pre-tokenised sentence. The crucial rule is that later model outputs must contain **exactly one annotation per input token**.

Core columns:

- `id`: stable sentence identifier
- `language`: language name
- `script`: script/writing system
- `domain`: source genre or collection
- `text`: original sentence
- `tokens`: JSON-encoded list of input tokens
- `gold_pos`: JSON-encoded list of UPOS tags, when available
- `gold_lemma`: JSON-encoded list of lemmas, when available
- `gold_features`: JSON-encoded list of feature dictionaries
- `split`: `fewshot`, `eval`, or `unlabeled`
- `notes`: reminder that this sample is illustrative

In [ ]:
def j(x):
    return json.dumps(x, ensure_ascii=False)

rows = [
    {
        'id': 'grc_001', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy',
        'text': 'λόγος ἐστὶ καλός .',
        'tokens': ['λόγος', 'ἐστὶ', 'καλός', '.'],
        'gold_pos': ['NOUN', 'AUX', 'ADJ', 'PUNCT'],
        'gold_lemma': ['λόγος', 'εἰμί', 'καλός', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Pres'},
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'grc_002', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy',
        'text': 'οἱ ἄνδρες γράφουσι .',
        'tokens': ['οἱ', 'ἄνδρες', 'γράφουσι', '.'],
        'gold_pos': ['DET', 'NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['ὁ', 'ἀνήρ', 'γράφω', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur', 'Gender': 'Masc'},
            {'Case': 'Nom', 'Number': 'Plur', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Pres'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'grc_003', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy_noisy',
        'text': 'βασιλεὺς εἶπεν λόγον .',
        'tokens': ['βασιλεὺς', 'εἶπεν', 'λόγον', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['βασιλεύς', 'λέγω', 'λόγος', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'xcl_001', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy',
        'text': 'այր մի եկն .',
        'tokens': ['այր', 'մի', 'եկն', '.'],
        'gold_pos': ['NOUN', 'NUM', 'VERB', 'PUNCT'],
        'gold_lemma': ['այր', 'մի', 'գալ', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'xcl_002', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy',
        'text': 'թագաւորն գրեաց նամակ .',
        'tokens': ['թագաւորն', 'գրեաց', 'նամակ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['թագաւոր', 'գրել', 'նամակ', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'xcl_003', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy_noisy',
        'text': 'աշակերտք ընթերցան գիրք .',
        'tokens': ['աշակերտք', 'ընթերցան', 'գիրք', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['աշակերտ', 'ընթեռնուլ', 'գիրք', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'oge_001', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy',
        'text': 'კაცი მოვიდა .',
        'tokens': ['კაცი', 'მოვიდა', '.'],
        'gold_pos': ['NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['კაცი', 'მოსვლა', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'oge_002', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy',
        'text': 'წიგნი კეთილი არს .',
        'tokens': ['წიგნი', 'კეთილი', 'არს', '.'],
        'gold_pos': ['NOUN', 'ADJ', 'AUX', 'PUNCT'],
        'gold_lemma': ['წიგნი', 'კეთილი', 'ყოფნა', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Pres'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'oge_003', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy_noisy',
        'text': 'მოწაფენი წერენ წიგნსა .',
        'tokens': ['მოწაფენი', 'წერენ', 'წიგნსა', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['მოწაფე', 'წერა', 'წიგნი', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Pres'},
            {'Case': 'Dat', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'syr_001', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy',
        'text': 'ܓܒܪܐ ܐܬܐ .',
        'tokens': ['ܓܒܪܐ', 'ܐܬܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['ܓܒܪܐ', 'ܐܬܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'syr_002', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy',
        'text': 'ܡܠܟܐ ܟܬܒ ܐܓܪܬܐ .',
        'tokens': ['ܡܠܟܐ', 'ܟܬܒ', 'ܐܓܪܬܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['ܡܠܟܐ', 'ܟܬܒ', 'ܐܓܪܬܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Number': 'Sing', 'Gender': 'Fem'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'syr_003', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy_noisy',
        'text': 'ܬܠܡܝܕܐ ܩܪܐ ܟܬܒܐ .',
        'tokens': ['ܬܠܡܝܕܐ', 'ܩܪܐ', 'ܟܬܒܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['ܬܠܡܝܕܐ', 'ܩܪܐ', 'ܟܬܒܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'eval'
    },
]

# Add metadata and serialise list columns for CSV.
for r in rows:
    r['notes'] = 'Toy pedagogical placeholder; replace with project-validated data.'
    for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
        r[col] = j(r[col])

df = pd.DataFrame(rows)
csv_path = DATA_DIR / 'toy_sentences.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'Wrote {len(df)} rows to {csv_path}')
df[['id', 'language', 'script', 'domain', 'text', 'split']]

In [ ]:
UPOS = [
    'ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN','NUM','PART','PRON','PROPN',
    'PUNCT','SCONJ','SYM','VERB','X'
]
FEATURES = ['Case', 'Number', 'Gender', 'Person', 'Tense', 'Mood', 'Voice']
CONFIDENCE = ['low', 'medium', 'high']

schema = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['sentence_id', 'language', 'tokens'],
    'properties': {
        'sentence_id': {'type': 'string'},
        'language': {'type': 'string'},
        'tokens': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['surface', 'lemma', 'upos', 'features', 'confidence', 'comment'],
                'properties': {
                    'surface': {'type': 'string'},
                    'lemma': {'type': ['string', 'null']},
                    'upos': {'type': 'string', 'enum': UPOS},
                    'features': {
                        'type': 'object',
                        'additionalProperties': {'type': ['string', 'null']}
                    },
                    'confidence': {'type': 'string', 'enum': CONFIDENCE},
                    'comment': {'type': ['string', 'null']}
                }
            }
        }
    }
}

schema_path = SCHEMA_DIR / 'pos_lemma_morph_schema.json'
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
print('Wrote schema to', schema_path)

## Next notebook

Continue with `01_prompting_zero_few_shot.ipynb`.